# 01 — BCRD Exchange Rates | Bronze Ingestion

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Ingest the official USD/DOP daily exchange-rate history published by the Central Bank of the Dominican Republic (BCRD) into the Bronze layer, preserving source values and adding ingestion metadata for traceability.

## Data Source (verified)
| Item | Value |
|---|---|
| Institution | Banco Central de la República Dominicana (BCRD) |
| File | `BCRD - Histórico diario de tasas.xlsx` |
| Sheet | `Histórico` (data) · `Calculo de apre-depre` (methodology notes) |
| Coverage | 20/9/2021 – 20/9/2026 (last observation 2026-09-18) |
| Layout | 8 header/notes rows; column headers on row 9; 1,253 data rows |
| Grain | One row per business day (buy and sell rate) |

## Lakehouse
- **Raw landing:** `/Volumes/rd_economic_intelligence/bronze/bcrd_raw/exchange_rates/`
- **Target table:** `rd_economic_intelligence.bronze.bcrd_exchange_rates_daily`
- **Load pattern:** idempotent per source file — rows previously loaded from the same file are replaced, never duplicated.

## Bronze rules
- Source values are **not** modified, corrected or discarded.
- Column names are converted to Delta-safe snake_case (the source headers contain `|`, `%` and spaces). The mapping is documented in this notebook.
- No business date is built here; `Año/Mes/Día` stay as delivered. Date construction belongs to Silver.
- Data-quality checks are **observational** in Bronze: they report, they do not filter.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
# Step 0 — Dependencies: serverless does not ship openpyxl, required by pandas to read .xlsx
dbutils.library.restartPython()

In [0]:
# ============================================================
# Step 1 — Configuration
# ============================================================
from datetime import datetime, timezone
import uuid

CATALOG = "rd_economic_intelligence"
SCHEMA = "bronze"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/bcrd_raw/exchange_rates/"
SOURCE_FILE = "BCRD - Histórico diario de tasas.xlsx"
SOURCE_SHEET = "Histórico"
HEADER_ROW_INDEX = 8          # 0-based: headers are on Excel row 9
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.bcrd_exchange_rates_daily"

# Source header -> Bronze column (Delta-safe). Order matches the source file.
COLUMN_MAPPING = {
    "Año": "anio",
    "Mes": "mes",
    "Día": "dia",
    "Compra": "compra",
    "Compra | Apre/depre acumulada (%)": "compra_apre_depre_acumulada_pct",
    "Compra | Apre/depre interanual (%)": "compra_apre_depre_interanual_pct",
    "Venta": "venta",
    "Venta | Apre/depre acumulada (%)": "venta_apre_depre_acumulada_pct",
    "Venta | Apre/depre interanual (%)": "venta_apre_depre_interanual_pct",
}

BATCH_ID = str(uuid.uuid4())
INGESTION_TS = datetime.now(timezone.utc)

print(f"Batch ID:     {BATCH_ID}")
print(f"Target table: {TARGET_TABLE}")

In [0]:
# ============================================================
# Step 2 — Raw landing verification
# ============================================================
files = {f.name: f for f in dbutils.fs.ls(RAW_PATH)}

print("Files found:")
for f in files.values():
    print(f"- {f.name} | {f.size:,} bytes")

if SOURCE_FILE not in files:
    raise FileNotFoundError(f"'{SOURCE_FILE}' not found in {RAW_PATH}")

source_info = files[SOURCE_FILE]
SOURCE_PATH = RAW_PATH + SOURCE_FILE
SOURCE_MODIFIED_TS = datetime.fromtimestamp(source_info.modificationTime / 1000, tz=timezone.utc)

In [0]:
# ============================================================
# Step 3 — Read the Excel file and capture its header metadata
# ============================================================
import pandas as pd

# Title block (rows 1-3) — keeps the declared coverage period for lineage
title_block = pd.read_excel(SOURCE_PATH, sheet_name=SOURCE_SHEET, header=None, nrows=3)
REPORT_TITLE = str(title_block.iloc[1, 0])
REPORT_PERIOD = str(title_block.iloc[2, 0])

pdf = pd.read_excel(SOURCE_PATH, sheet_name=SOURCE_SHEET, header=HEADER_ROW_INDEX)

print(f"Report title:  {REPORT_TITLE}")
print(f"Report period: {REPORT_PERIOD}")
print(f"Rows read:     {len(pdf):,}")
print(f"Columns read:  {list(pdf.columns)}")

In [0]:
# ============================================================
# Step 4 — Structural contract: fail fast if the layout changed
# ============================================================
expected = list(COLUMN_MAPPING.keys())
actual = [str(c).strip() for c in pdf.columns]

missing = [c for c in expected if c not in actual]
unexpected = [c for c in actual if c not in expected]

if missing or unexpected:
    raise ValueError(
        f"Source layout changed. Missing: {missing} | Unexpected: {unexpected}"
    )
if pdf.empty:
    raise ValueError("Source sheet contains no data rows.")

print("PASS: source layout matches the expected contract")

In [0]:
# ============================================================
# Step 5 — Add lineage metadata and build the Spark DataFrame
# ============================================================
from pyspark.sql import functions as F

pdf.columns = actual
pdf = pdf.rename(columns=COLUMN_MAPPING)

# Excel row number of each record (header on row 9 -> first data row is 10)
pdf["_source_row_number"] = range(HEADER_ROW_INDEX + 2, HEADER_ROW_INDEX + 2 + len(pdf))

df = (
    spark.createDataFrame(pdf)
    .withColumn("_source_file", F.lit(SOURCE_FILE))
    .withColumn("_source_path", F.lit(SOURCE_PATH))
    .withColumn("_source_sheet", F.lit(SOURCE_SHEET))
    .withColumn("_source_file_size_bytes", F.lit(source_info.size).cast("long"))
    .withColumn("_source_modified_ts", F.lit(SOURCE_MODIFIED_TS).cast("timestamp"))
    .withColumn("_report_period", F.lit(REPORT_PERIOD))
    .withColumn("_ingestion_ts", F.lit(INGESTION_TS).cast("timestamp"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

df.printSchema()
display(df.limit(5))

In [0]:
# ============================================================
# Step 6 — Observational data-quality checks (report only)
# ============================================================
date_col = F.make_date("anio", "mes", "dia")

checks = {
    "row_count": df.count(),
    "null_year_month_day": df.filter(F.col("anio").isNull() | F.col("mes").isNull() | F.col("dia").isNull()).count(),
    "invalid_calendar_date": df.filter(date_col.isNull()).count(),
    "duplicate_dates": df.groupBy("anio", "mes", "dia").count().filter("count > 1").count(),
    "null_rates": df.filter(F.col("compra").isNull() | F.col("venta").isNull()).count(),
    "non_positive_rates": df.filter((F.col("compra") <= 0) | (F.col("venta") <= 0)).count(),
    "buy_greater_than_sell": df.filter(F.col("compra") > F.col("venta")).count(),
}

print("Data-quality results (Bronze is observational — no rows were removed):")
for name, value in checks.items():
    status = "INFO" if name == "row_count" else ("PASS" if value == 0 else "WARN")
    print(f"  {status:<4} {name}: {value:,}")

In [0]:
# ============================================================
# Step 7 — Idempotent write to the Bronze Delta table
# ============================================================
table_exists = spark.catalog.tableExists(TARGET_TABLE)

if table_exists:
    # Replace rows from this same source file so re-runs never duplicate data
    spark.sql(f"DELETE FROM {TARGET_TABLE} WHERE _source_file = '{SOURCE_FILE}'")

(
    df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "false")
    .saveAsTable(TARGET_TABLE)
)

if not table_exists:
    spark.sql(f"""
        COMMENT ON TABLE {TARGET_TABLE} IS
        'Bronze: BCRD official USD/DOP daily buy/sell rates, source-preserving, with ingestion lineage metadata.'
    """)

print(f"Written to {TARGET_TABLE}")

In [0]:
# ============================================================
# Step 8 — Post-load verification
# ============================================================
summary = spark.sql(f"""
    SELECT
        COUNT(*)                                     AS total_rows,
        COUNT(DISTINCT _batch_id)                    AS batches,
        MIN(make_date(anio, mes, dia))               AS first_date,
        MAX(make_date(anio, mes, dia))               AS last_date,
        MAX(_ingestion_ts)                           AS last_ingestion_ts
    FROM {TARGET_TABLE}
    WHERE _source_file = '{SOURCE_FILE}'
""")
display(summary)

loaded = summary.first()["total_rows"]
assert loaded == checks["row_count"], f"Row count mismatch: read {checks['row_count']}, loaded {loaded}"
print(f"PASS: {loaded:,} rows loaded — matches rows read from source")

In [0]:
# Return a run summary (visible in Jobs run output when executed as a task)
import json
row = summary.first()
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "target_table": TARGET_TABLE,
    "batch_id": BATCH_ID,
    "rows_loaded": int(row["total_rows"]),
    "first_date": str(row["first_date"]),
    "last_date": str(row["last_date"]),
    "report_period": REPORT_PERIOD,
    "dq_checks": checks,
}))